# autoprof

> Functionality for running autoprof to measure the ICL in images of galaxy clusters.

In [1]:
# | default_exp autoprof

In [12]:
import glob
import numpy as np
from astropy.io import fits

import importlib.util
import sys
import os
import traceback
from pathlib import Path
from textwrap import dedent

import autoprof
from autoprof import Pipeline 


In [13]:
# | hide

In [14]:
# | export

def clean_and_process_images_for_autoprof(
    image_directory,
    mode="bulk",
    clusterid=None,
    bands=None,
    nan_value_replace='extreme',
    output_directory=None,
    boxsize=300
):
    """
    Cleans and processes images by replacing NaNs and saving to output directory.
    
    Parameters:
    - imdir (str): Directory where images are located.
    - mode (str): 'bulk' or 'single'. 'bulk' processes all matching images, 'single' targets one cluster.
    - clusterid (str): Required if mode is 'single'.
    - bands (list): List of bands to process. Default is ["H", "J", "Y", "HJY"].
    - nan_value_replace (str): 'extreme' to replace with 99, 'nanmin' to replace with np.nanmin.
    - output_directory (str or Path): Optional path to save cleaned images.
    - boxsize (int): Box size used in filename pattern.
    """
    if bands is None:
        bands = ["H", "J", "Y", "HJY"]

    for band in bands:
        if mode == "single":
            if not clusterid:
                raise ValueError("clusterid must be provided in single mode.")
            # pattern = f"{image_directory}{clusterid}_{band}_BKGSUB_boxsize_{boxsize}.fits"
            pattern = image_directory / f"EUC_NIR_W-STK_{band}-{clusterid}.fits"
            
        elif mode == "bulk":
            pattern = image_directory / f"EUC_NIR_W-STK_{band}-*.fits"
            
        else:
            raise ValueError("mode must be either 'bulk' or 'single'")

        image_files = sorted(glob.glob(str(pattern)))
        
        if not image_files:
            print(f"No files matched pattern: {pattern}")
            continue

        for image in image_files:
            image_path = Path(image)
            print(f"\nProcessing Image: {image_path.name}")
            
            new_clean_image_name = f"cleaned_{image_path.name}"

            with fits.open(image_path) as hdul:
                image_data = hdul[1].data.copy()
                image_header = hdul[1].header.copy()
                nan_mask = np.isnan(image_data)
                print(f"Number of NaN values: {nan_mask.sum()}")

                clean_image_data = image_data.copy()
                if nan_value_replace == 'extreme':
                    print("Replacing NaNs with 99.")
                    clean_image_data[nan_mask] = 99
                elif nan_value_replace == 'nanmin':
                    print("Replacing NaNs with nanmin.")
                    clean_image_data[nan_mask] = np.nanmin(clean_image_data)
                else:
                    raise ValueError("nan_value_replace must be 'extreme' or 'nanmin'.")

            save_dir = Path(output_directory) if output_directory else image_path.parent
            save_path = save_dir / new_clean_image_name
            save_dir.mkdir(parents=True, exist_ok=True)

            fits.writeto(save_path, clean_image_data, header=image_header, overwrite=True)
            print(f"Saved cleaned image to: {save_path}")

In [5]:
# | export

def run_autoprof(
    ids,
    image_files,
    mask_files=None,
    mode="image list",
    unit_type="intensity",
    gscale=0.4,
    pixelscale=0.3,
    zeropoint=23.9,
    out_dir="./",
    config_name="basic_config.py",
    log_path="AutoProf.log",
    fourier_orders=None,
    forced_photometry=False,
    forced_profile_filter=None,
    forced_profile_path=None, # .prof file path "dir/x.prof"
):
    """
    Running AutoProf with optional fixed photometry.

    If forced_photometry=True, the reference profile in forced_profile_path is used,
    and the pipeline steps are adjusted accordingly.
    """

    os.makedirs(out_dir, exist_ok=True)
    config_file = os.path.join(out_dir, config_name)

    # Modified pipeline based on forced photometry
    if forced_photometry:
        if not forced_profile_path:
            raise ValueError("forced_profile_path must be provided when forced_photometry=True.")
        if not forced_profile_filter:
            raise ValueError("forced_profile_filter must be provided when forced_photometry=True.")

        pipeline_steps = [
            "background",
            "psf",
            "center forced",
            "isophoteinit forced",
            "isophotefit forced",
            "isophoteextract forced",
            "writeprof",
        ]

        # Adding "_fp"+forced_profile_filter to each ID for forced photometry
        ids = [f"{_id}_fp_{forced_profile_filter}" for _id in ids]
    else:
        pipeline_steps = [
            "background",
            "psf",
            "center",
            "isophoteinit",
            "isophotefit",
            "isophoteextract",
            "writeprof",
        ]

    # Adding masking if file provided
    if mask_files is not None:
        pipeline_steps.insert(0, "mask segmentation map")

    # Writing AutoProf config
    with open(config_file, "w") as f:
        f.write(dedent(f"""\
import numpy as np
ap_process_mode = "{mode}"
ap_name = {ids}
ap_image_file = {image_files}
"""))
        if mask_files is not None:
            f.write(f"ap_mask_file = {mask_files}\n")

        f.write(dedent(f"""\
ap_saveto = "{out_dir}"
ap_pixscale = {pixelscale}
ap_zeropoint = {zeropoint}
ap_samplegeometricscale = {gscale}
ap_doplot = True
ap_extractfull = True
ap_fluxunits = "{unit_type}"
ap_isoclip = True
ap_isoclip_nsigma = 5
ap_ellipsefit = True
ap_fix_pa = False
ap_initial_pa = 45.0 
ap_fix_ellipticity = False
ap_initial_ellipticity = 0.3
"""))

        if forced_photometry:
            f.write(f'ap_forcing_profile = r"{forced_profile_path}"\n')

        if fourier_orders:
            f.write(f"ap_iso_measurecoefs = {fourier_orders}\n")

        f.write(f"ap_new_pipeline_steps = {pipeline_steps}\n")

    # Running AutoProf
    os.chdir(out_dir)
    try:
        import sys
        if config_name.replace(".py", "") in sys.modules:
            del sys.modules[config_name.replace(".py", "")]

        PIPELINE = Pipeline.Isophote_Pipeline(loggername=log_path)
        PIPELINE.Process_ConfigFile(config_name.replace(".py", ""))

        print(f"AutoProf completed successfully for {ids}!")

    except Exception as e:
        print(f"AutoProf failed for {ids}. Logging error...")

        error_log_file = Path(out_dir) / f"autoprof_error_{ids[0]}.log"
        with open(error_log_file, "w") as log:
            log.write(f"AutoProf failed for {ids}\n")
            log.write(f"Error type: {type(e).__name__}\n")
            log.write(f"Error message: {str(e)}\n")
            log.write("Full traceback:\n")
            log.write(traceback.format_exc())

        print(f"Error log saved to: {error_log_file}")

        
        

## The function that combines image prepping and running autoprof.
> Running the function below will create a "autoprof_results" folder in each cluster within cluster list provided.

> The prepped images will be contained as "cleaned" within the same autoprof_results folder.

> The autoprof will access the "cleaned" image from autoprof_results folder and mask from cluster_directory. The resultant diagnostic plots and profile files will be then stored in the created autoprof_results folder.

In [15]:

def process_cluster_with_autoprof(
    image_dir, 
    cluster_directory, 
    cluster_ids, 
    filters, 
    mask_filter=None,
    prefix=None,
    forced_photometry=False,
    forced_profile_filter=None,
    forced_profile_path=None
):

    image_dir = Path(image_dir)

    for cluster_id in cluster_ids:
        cluster_path = Path(cluster_directory) / cluster_id
        AP_results_dir = cluster_path / "autoprof_results"
        AP_results_dir.mkdir(parents=True, exist_ok=True)

        for filter in filters:
            actual_mask_filter = mask_filter if mask_filter is not None else filter

            # Constructing IDs
            if actual_mask_filter != filter:
                cluster_label = f"{cluster_id}_{filter}_{actual_mask_filter}"
            else:
                cluster_label = f"{cluster_id}_{filter}"

            if prefix:
                cluster_label += f"_{prefix}"

            # Step 1: Clean image
            clean_and_process_images_for_autoprof(
                image_directory=image_dir,
                output_directory=AP_results_dir,
                mode='single',
                clusterid=cluster_id,
                bands=[filter]
            )

            image_filename = f"cleaned_EUC_NIR_W-STK_{filter}-{cluster_id}.fits"
            image_path = AP_results_dir / image_filename

            mask_filename = f"{cluster_id}_{actual_mask_filter}_measurement_mask.fits"
            mask_path = cluster_path / mask_filename

            #  Run AutoProf
            run_autoprof(
                ids=[cluster_label],
                image_files=[str(image_path)],
                mask_files=[str(mask_path)],
                out_dir=str(AP_results_dir),
                fourier_orders=(1, 4),
                forced_photometry=forced_photometry,
                forced_profile_filter = forced_profile_filter,
                forced_profile_path=str(forced_profile_path) if forced_photometry else None
            )

            # Clean unnecessary files
            always_keep = {"basic_config.py"}
            jpg_keywords_to_keep = [
                "Background_hist_",
                "mask_",
                "initialize_ellipse_",
                "photometry_",
                "photometry_ellipse_",
            ]
            valid_suffixes_to_keep = [".prof", ".aux"]

            for f in AP_results_dir.iterdir():
                if f.is_file():
                    name = f.name
                    if name in always_keep:
                        continue
                    if any(name.endswith(suffix) for suffix in valid_suffixes_to_keep):
                        continue
                    if name.endswith(".jpg") and any(kw in name for kw in jpg_keywords_to_keep):
                        continue
                    try:
                        f.unlink()
                        print(f"Deleted: {name}")
                    except Exception as e:
                        print(f"Could not delete {name}: {e}")


# Example to run the image preparation and autoprof with existing mask file in output directory.

In [18]:
imdir = f"/home/ppztk1/euclid_data/Q1_R1_clusters_v0.7/tutku/"
outdir = f"/home/ppztk1/Erosita/Outputs_Clusters/"

clids = ["1eRASS J035423.6-475145"]
filters = ["H"] 

process_cluster_with_autoprof(image_dir=imdir, 
                              cluster_directory=outdir, 
                              cluster_ids=clids, 
                              filters=filters)
